# Feedforward Neural Network for Financial Market Direction Prediction


This practical extends Practical 1 (the single-layer Perceptron) to a
**Feedforward Neural Network (FFNN / Multi-Layer Perceptron)** implemented
in **PyTorch**, applied to the same kind of problem: predicting whether a
stock's next-day return will be positive or negative from today's
technical indicators.

In [1]:
# ---------------------------------------------------------
# Feedforward Neural Network for Financial Market Direction
# Prediction
#
# Objective:
# Predict whether a stock is expected to have a
# positive or negative return on the next trading day,
# using a Feedforward Neural Network (Multi-Layer Perceptron)
# instead of a single-layer Perceptron.
#
# Note:
# This is a simple educational example using a synthetic
# dataset to demonstrate how a feedforward neural network
# learns non-linear relationships that a plain Perceptron
# (Practical 1) cannot.
#
# Framework: PyTorch
# ---------------------------------------------------------

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cu130


In [2]:
# ---------------------------------------------------------
# Why move from a Perceptron to a Feedforward Neural Network?
#
# In Practical 1, the Perceptron:
#
#   z = w.X + b
#   output = step(z)
#
# can only draw a single straight line (a linear decision
# boundary) between "positive return" and "negative return"
# samples.
#
# Financial markets, however, are driven by non-linear and
# interacting effects between indicators. For example:
#
#   High RSI (overbought) + High Volume Change
#   may signal a reversal (negative return),
#
# while
#
#   Moderate RSI + High Volume Change
#   may signal continuation (positive return).
#
# A single straight line cannot separate patterns like this.
#
# A Feedforward Neural Network (FFNN) stacks multiple layers
# of neurons with non-linear activation functions
# (e.g. ReLU, Sigmoid), allowing it to learn curved,
# non-linear decision boundaries.
#
#         Input Layer -> Hidden Layer(s) -> Output Layer
#      (indicators)      (non-linear         (probability of
#                          feature learning)   positive return)
# ---------------------------------------------------------

In [3]:
# ---------------------------------------------------------
# Financial Dataset (Synthetic)
#
# Features:
#
# Daily Return (%)
# -> Percentage change in stock price today.
#
# RSI (Relative Strength Index)
# -> Measures market momentum (0-100).
#    RSI > 70  -> Overbought
#    RSI < 30  -> Oversold
#
# Volume Change (%)
# -> Percentage increase/decrease in trading volume
#    compared to its recent average.
#
# MACD Signal
# -> Moving Average Convergence Divergence value.
#    Positive -> bullish momentum, Negative -> bearish momentum.
#
# Volatility (%)
# -> Rolling standard deviation of recent returns.
#    Higher volatility -> more uncertain/risky conditions.
#
#
# Label
#
# 1 -> Tomorrow's return was positive.
# 0 -> Tomorrow's return was negative.
#
# IMPORTANT:
#
# These indicators do not truly determine tomorrow's return.
# This dataset is SYNTHETIC and generated with a hidden
# non-linear rule (interactions + noise) purely so that we
# can demonstrate why a Feedforward Neural Network outperforms
# a single-layer Perceptron on non-linearly separable data.
#
# In real quantitative finance, labels like this are
# engineered from historical price/volume data, and the
# true relationship between indicators and future returns
# is unknown, weak, and noisy.
# ---------------------------------------------------------

n_samples = 800

daily_return   = np.random.normal(0, 1.5, n_samples)          # %
rsi            = np.random.uniform(10, 90, n_samples)         # 0-100
volume_change  = np.random.normal(0, 20, n_samples)           # %
macd           = np.random.normal(0, 1.0, n_samples)          # signal
volatility     = np.random.uniform(0.5, 5.0, n_samples)       # %

# ---------------------------------------------------------
# Hidden, NON-LINEAR rule used only to generate labels.
# The model never sees this rule -- it must discover a
# similar pattern purely from the features and labels.
#
# Rule (deliberately non-linear / interaction-based):
#
#   score = macd
#         + 0.03 * daily_return * volume_change      (interaction)
#         - 0.02 * (rsi - 50) ** 2 / 10               (overbought/oversold curve)
#         - 0.05 * volatility ** 2                    (risk penalty, non-linear)
#         + noise
#
#   label = 1 if score > 0 else 0
# ---------------------------------------------------------

noise = np.random.normal(0, 1.0, n_samples)

score = (
    macd
    + 0.03 * daily_return * volume_change
    - 0.02 * ((rsi - 50) ** 2) / 10
    - 0.05 * (volatility ** 2)
    + noise
)

labels = (score > 0).astype(np.float32)

X = np.column_stack([daily_return, rsi, volume_change, macd, volatility]).astype(np.float32)
y = labels

print("Feature matrix shape:", X.shape)
print("Label vector shape  :", y.shape)
print("Class balance (1=Positive Return): {:.1f}% positive".format(100 * y.mean()))

Feature matrix shape: (800, 5)
Label vector shape  : (800,)
Class balance (1=Positive Return): 20.1% positive


In [4]:
# ---------------------------------------------------------
# Train / Test Split and Feature Standardization
#
# Neural networks train more reliably when input features
# are on a similar scale. Here we standardize each feature
# to zero mean and unit variance (z-score), using statistics
# from the TRAINING set only, then apply the same
# transformation to the test set.
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

feature_mean = X_train.mean(axis=0)
feature_std  = X_train.std(axis=0) + 1e-8

X_train_scaled = (X_train - feature_mean) / feature_std
X_test_scaled  = (X_test  - feature_mean) / feature_std

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("Training samples:", X_train_t.shape[0])
print("Test samples    :", X_test_t.shape[0])

Training samples: 640
Test samples    : 160


In [5]:
# ---------------------------------------------------------
# Feedforward Neural Network (Multi-Layer Perceptron)
#
# Architecture:
#
#   Input Layer   : 5 neurons  (one per financial indicator)
#   Hidden Layer 1: 16 neurons, ReLU activation
#   Hidden Layer 2: 8  neurons, ReLU activation
#   Output Layer  : 1  neuron,  Sigmoid activation
#                   (probability that tomorrow's return
#                    is positive)
#
# Unlike the Perceptron (Practical 1), which has only an
# input and an output layer connected by a single set of
# weights, this network has HIDDEN LAYERS with non-linear
# activation functions. This lets it combine indicators in
# non-linear ways (e.g. RSI * Volume, MACD^2, etc.) and
# learn curved decision boundaries.
# ---------------------------------------------------------

class FeedforwardNN(nn.Module):

    def __init__(self, input_size):
        super(FeedforwardNN, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 16),   # Input -> Hidden Layer 1
            nn.ReLU(),                   # Non-linear activation

            nn.Linear(16, 8),            # Hidden Layer 1 -> Hidden Layer 2
            nn.ReLU(),                   # Non-linear activation

            nn.Linear(8, 1),             # Hidden Layer 2 -> Output
            nn.Sigmoid()                 # Squashes output to (0, 1)
                                          # interpreted as P(positive return)
        )

    def forward(self, x):
        return self.network(x)


# Instantiate the model
model = FeedforwardNN(input_size=X_train_t.shape[1])
print(model)

FeedforwardNN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [6]:
# ---------------------------------------------------------
# Loss Function and Optimizer
#
# Binary Cross-Entropy Loss (BCELoss)
# -> Standard loss function for binary classification
#    problems (Positive Return vs Negative Return).
#
# Adam Optimizer
# -> Adaptive gradient-based optimizer that updates the
#    network's weights using backpropagation. This replaces
#    the manual weight-update rule used in the Perceptron
#    (self.weights += lr * error * xi) with automatic
#    differentiation ("autograd") across all layers.
# ---------------------------------------------------------

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [7]:
# ---------------------------------------------------------
# Training
#
# Steps (repeated every epoch):
#
# 1. Forward pass  -> compute predictions for all training
#                      samples.
# 2. Compute loss   -> compare predictions with actual
#                      labels using BCELoss.
# 3. Backward pass  -> compute gradients of the loss with
#                      respect to every weight in the
#                      network (backpropagation).
# 4. Update weights -> optimizer adjusts all weights to
#                      reduce the loss.
#
# This mirrors the Perceptron's train() method from
# Practical 1, but backpropagation now updates weights
# across MULTIPLE layers instead of a single layer.
# ---------------------------------------------------------

epochs = 200
loss_history = []

for epoch in range(epochs):

    model.train()

    # 1. Forward pass
    predictions = model(X_train_t)

    # 2. Compute loss
    loss = criterion(predictions, y_train_t)

    # 3. Backward pass (backpropagation)
    optimizer.zero_grad()
    loss.backward()

    # 4. Update weights
    optimizer.step()

    loss_history.append(loss.item())

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch + 1:3d}/{epochs}]  Loss: {loss.item():.4f}")

Epoch [ 20/200]  Loss: 0.4724
Epoch [ 40/200]  Loss: 0.3231
Epoch [ 60/200]  Loss: 0.2508
Epoch [ 80/200]  Loss: 0.2149
Epoch [100/200]  Loss: 0.2018
Epoch [120/200]  Loss: 0.1901
Epoch [140/200]  Loss: 0.1799
Epoch [160/200]  Loss: 0.1686


Epoch [180/200]  Loss: 0.1594
Epoch [200/200]  Loss: 0.1512


In [1]:
# ---------------------------------------------------------
# Training Loss Curve
#
# A steadily decreasing loss shows the network is learning
# to separate positive-return days from negative-return
# days using the training data.
# ---------------------------------------------------------

import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(loss_history, color="#1f77b4", linewidth=1.8)
plt.title("Training Loss over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'loss_history' is not defined

<Figure size 700x400 with 0 Axes>

In [9]:
# ---------------------------------------------------------
# Evaluation on Held-Out Test Data
#
# The network has never seen the test samples during
# training. Evaluating on this unseen data tells us how
# well it is likely to generalize to new, future trading
# days -- not just memorize the training set.
# ---------------------------------------------------------

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()

with torch.no_grad():
    train_probs = model(X_train_t)
    test_probs  = model(X_test_t)

    train_preds = (train_probs >= 0.5).float()
    test_preds  = (test_probs  >= 0.5).float()

train_acc = accuracy_score(y_train_t, train_preds)
test_acc  = accuracy_score(y_test_t, test_preds)

print(f"Training Accuracy : {train_acc * 100:.2f}%")
print(f"Test Accuracy     : {test_acc * 100:.2f}%\n")

print("Confusion Matrix (Test Set)")
print("Rows = Actual, Columns = Predicted [Negative, Positive]")
print(confusion_matrix(y_test_t, test_preds))

print("\nClassification Report (Test Set)")
print(classification_report(
    y_test_t, test_preds,
    target_names=["Negative Return (0)", "Positive Return (1)"]
))

Training Accuracy : 93.59%
Test Accuracy     : 83.75%

Confusion Matrix (Test Set)
Rows = Actual, Columns = Predicted [Negative, Positive]
[[116  12]
 [ 14  18]]

Classification Report (Test Set)
                     precision    recall  f1-score   support

Negative Return (0)       0.89      0.91      0.90       128
Positive Return (1)       0.60      0.56      0.58        32

           accuracy                           0.84       160
          macro avg       0.75      0.73      0.74       160
       weighted avg       0.83      0.84      0.84       160



In [10]:
# ---------------------------------------------------------
# Predictions
#
# If prediction = 1
#
# Expected Positive Return
#
# If prediction = 0
#
# Expected Negative Return
#
# We also print the model's estimated PROBABILITY, which a
# single-layer Perceptron cannot provide (it only outputs a
# hard 0/1 decision via the step function).
# ---------------------------------------------------------

print("Sample Predictions (first 10 test samples)\n")

feature_names = ["Daily Return (%)", "RSI", "Volume Change (%)", "MACD", "Volatility (%)"]

with torch.no_grad():
    sample_X = X_test_t[:10]
    sample_probs = model(sample_X).squeeze().numpy()
    sample_preds = (sample_probs >= 0.5).astype(int)

sample_X_raw = X_test[:10]  # original, unscaled values for readability

for i in range(10):
    print("Features:")
    for name, value in zip(feature_names, sample_X_raw[i]):
        print(f"   {name:<20s}: {value:>8.2f}")

    prediction = sample_preds[i]
    probability = sample_probs[i]

    if prediction == 1:
        print(f"Prediction  : Expected Positive Return  (confidence: {probability*100:.1f}%)\n")
    else:
        print(f"Prediction  : Expected Negative Return  (confidence: {(1-probability)*100:.1f}%)\n")

Sample Predictions (first 10 test samples)

Features:
   Daily Return (%)    :    -2.11
   RSI                 :    79.66
   Volume Change (%)   :    14.84
   MACD                :     0.09
   Volatility (%)      :     0.93
Prediction  : Expected Negative Return  (confidence: 99.0%)

Features:
   Daily Return (%)    :     0.10
   RSI                 :    77.05
   Volume Change (%)   :     5.74
   MACD                :     0.31
   Volatility (%)      :     2.28
Prediction  : Expected Negative Return  (confidence: 99.2%)

Features:
   Daily Return (%)    :     1.24
   RSI                 :    30.61
   Volume Change (%)   :   -10.72
   MACD                :     0.42
   Volatility (%)      :     3.17
Prediction  : Expected Negative Return  (confidence: 99.4%)

Features:
   Daily Return (%)    :     0.52
   RSI                 :    53.30
   Volume Change (%)   :    -5.63
   MACD                :     1.53
   Volatility (%)      :     3.69
Prediction  : Expected Negative Return  (confidence: 

In [11]:
# ---------------------------------------------------------
# Perceptron (Practical 1) vs Feedforward Neural Network
#
# To make the improvement concrete, we train a plain
# single-layer Perceptron (like Practical 1) on the SAME
# standardized data and compare its test accuracy against
# the Feedforward Neural Network above.
# ---------------------------------------------------------

def step_function(x):
    return 1 if x >= 0 else 0

class Perceptron:

    def __init__(self, input_size, learning_rate=0.01):
        self.weights = np.zeros(input_size)
        self.bias = 0
        self.lr = learning_rate

    def predict(self, x):
        z = np.dot(x, self.weights) + self.bias
        return step_function(z)

    def train(self, X, y, epochs=25):
        for epoch in range(epochs):
            for xi, target in zip(X, y):
                prediction = self.predict(xi)
                error = target - prediction
                self.weights += self.lr * error * xi
                self.bias += self.lr * error


perceptron = Perceptron(input_size=X_train_scaled.shape[1])
perceptron.train(X_train_scaled, y_train, epochs=25)

perceptron_preds = np.array([perceptron.predict(xi) for xi in X_test_scaled])
perceptron_acc = accuracy_score(y_test, perceptron_preds)

print(f"Perceptron (Practical 1) Test Accuracy         : {perceptron_acc * 100:.2f}%")
print(f"Feedforward Neural Network (Practical 2) Test Accuracy : {test_acc * 100:.2f}%")
print("\nBecause the labels were generated from a NON-LINEAR rule")
print("(interaction terms + squared terms), the FFNN's hidden layers")
print("are able to capture patterns that the single-layer Perceptron's")
print("linear decision boundary cannot.")

Perceptron (Practical 1) Test Accuracy         : 76.88%
Feedforward Neural Network (Practical 2) Test Accuracy : 83.75%

Because the labels were generated from a NON-LINEAR rule
(interaction terms + squared terms), the FFNN's hidden layers
are able to capture patterns that the single-layer Perceptron's
linear decision boundary cannot.


In [12]:
# ---------------------------------------------------------
# Limitations
#
# 1. Still assumes today's indicators alone predict
#    tomorrow's return -- ignores longer historical
#    sequences (no memory of past days).
#
# 2. Prone to overfitting on small / noisy financial
#    datasets if the network is too large or trained too
#    long.
#
# 3. Requires careful feature scaling and hyperparameter
#    tuning (layers, neurons, learning rate, epochs).
#
# 4. Treats each day independently -- cannot model
#    momentum, trends, or regime shifts over time.
#
# 5. Real financial markets are highly noisy and
#    non-stationary (statistical properties change over
#    time), which limits any model's real-world accuracy.
#
# 6. Being a "black box", it is harder to interpret than
#    simpler models like Logistic Regression.
# ---------------------------------------------------------

In [13]:
# ---------------------------------------------------------
# Better Alternatives (for real-world quantitative finance)
#
# Logistic Regression
# -> Simple, interpretable linear probabilistic classifier.
#
# Random Forest
# -> Ensemble of decision trees, robust to noisy features.
#
# XGBoost / LightGBM
# -> Industry-standard gradient boosting models for
#    structured/tabular financial datasets.
#
# Deeper / Regularized MLPs
# -> This same FFNN with Dropout, Batch Normalization, and
#    L2 regularization to reduce overfitting.
#
# LSTM / GRU (Recurrent Neural Networks)
# -> Designed for financial TIME-SERIES data; retain memory
#    of past sequences of prices/indicators.
#
# Temporal Convolutional Networks / Transformer Models
# -> Capture long-range temporal dependencies across many
#    trading days.
#
# Reinforcement Learning
# -> Learns full trading STRATEGIES (buy/hold/sell actions)
#    rather than just predicting next-day direction.
# ---------------------------------------------------------

## Summary

| Aspect | Perceptron (Practical 1) | Feedforward Neural Network (Practical 2) |
|---|---|---|
| Layers | Input → Output only | Input → Hidden → Hidden → Output |
| Activation | Step function | ReLU (hidden), Sigmoid (output) |
| Decision boundary | Linear only | Non-linear |
| Output | Hard 0/1 | Probability (0–1) |
| Training | Manual weight update rule | Backpropagation + Adam optimizer |
| Captures feature interactions | No | Yes |

The Feedforward Neural Network generalizes the Perceptron by stacking
multiple layers with non-linear activations, allowing it to model the
kind of complex, interacting patterns (momentum × volume, overbought/oversold
curves, volatility effects) that are common — and still imperfect proxies
for reality — in quantitative finance.